# 第 10 章: ロジスティック回帰とアンサンブル学習の探索と可視化

ロジスティック回帰の損失の推移と、モデル別の特徴量の重要度を確認する。

In [ ]:
import sys

sys.path.append("..")

import pandas as pd
from japanese_font import use_japanese_font
from sklearn.ensemble import RandomForestClassifier

from lib.chapter02.iris_preprocessing import prepare_iris
from lib.chapter03.decision_tree import DecisionTree
from lib.chapter10.classifier import evaluate
from lib.chapter10.feature_importance import forest_importances, tree_importances
from lib.chapter10.logistic_regression import LogisticRegression
from lib.chapter10.random_forest import RandomForest
from lib.dataset import data_dir

use_japanese_font();

In [ ]:
split = prepare_iris(data_dir() / "iris.csv", test_size=0.3, seed=0)

## ロジスティック回帰の損失の推移

In [ ]:
logistic = LogisticRegression().fit(split.x_train, split.t_train)
losses = pd.Series(logistic.losses, name="交差エントロピー")
losses.plot(logy=True, title="勾配降下法の繰り返し回数と損失")
losses.iloc[[0, 9, 99, 999, 4999]].round(4)

In [ ]:
pd.DataFrame(
    logistic.weights, index=split.x_train.columns, columns=logistic.classes
).round(2)

## モデル別の特徴量の重要度

In [ ]:
tree = DecisionTree(max_depth=3).fit(split.x_train, split.t_train)
forest = RandomForest(n_estimators=100, max_features=2, seed=0)
forest.fit(split.x_train, split.t_train)
library = RandomForestClassifier(n_estimators=100, random_state=0)
library.fit(split.x_train, split.t_train)

importances = pd.DataFrame(
    {
        "決定木（深さ 3）": tree_importances(tree.tree, split.x_train, split.t_train),
        "ランダムフォレスト（自作）": forest_importances(
            forest, split.x_train, split.t_train
        ),
        "ランダムフォレスト（scikit-learn）": pd.Series(
            library.feature_importances_, index=split.x_train.columns
        ),
    }
)
importances.plot.barh(title="モデル別の特徴量の重要度")
importances.round(4)

## 森の大きさと正解率

In [ ]:
rows = []
for n_estimators in [1, 5, 10, 25, 50, 100]:
    score = evaluate(
        RandomForest(n_estimators=n_estimators, max_features=2, seed=0), split
    )
    rows.append(
        {"木の数": n_estimators, "訓練データ": score.train, "テストデータ": score.test}
    )
sizes = pd.DataFrame(rows).set_index("木の数")
sizes.plot(marker="o", title="ランダムフォレストの木の数と正解率")
sizes.round(4)